<a href="https://colab.research.google.com/github/ThialaMD/10_Scientific-Question/blob/main/Depression_Warning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# The '!' triggers a Linux bash terminal command to unzip the archive in Colab's VM
# Unzip your archive silently into the active workspace
!unzip -q /content/DI-Connect.zip

In [ ]:
# Run this cell to see your actual available data ranges and columns
print("=== DATA TIMELINE DIAGNOSTIC ===")
print(f"First recorded date in data: {df_master['date'].min().strftime('%Y-%m-%d')}")
print(f"Last recorded date in data:  {df_master['date'].max().strftime('%Y-%m-%d')}")

print("\n=== CHECKING SAMPLES PER CALIDRATION WINDOW ===")
dep_count = len(df_master[(df_master['date'] >= '2024-10-30') & (df_master['date'] <= '2025-03-31')])
hlth_count = len(df_master[(df_master['date'] >= '2026-04-01') & (df_master['date'] <= '2026-06-30')])
print(f"Days found in your Depressed Window: {dep_count}")
print(f"Days found in your Healthy Window:   {hlth_count}")

print("\n=== ACTUAL AVAILABLE METRICS ===")
print(df_master[['steps', 'total_sleep_minutes']].describe())

=== DATA TIMELINE DIAGNOSTIC ===
First recorded date in data: 2024-10-01
Last recorded date in data:  2026-06-30

=== CHECKING SAMPLES PER CALIDRATION WINDOW ===
Days found in your Depressed Window: 153
Days found in your Healthy Window:   91

=== ACTUAL AVAILABLE METRICS ===
              steps  total_sleep_minutes
count    638.000000           638.000000
mean    7354.568763           497.816684
std     2724.358693            57.353741
min     1557.803243           371.600148
25%     5524.812483           459.386540
50%     7956.636017           486.803172
75%     9467.000433           524.332715
max    12520.429560           682.229558


In [ ]:
import pandas as pd
import numpy as np

def run_theoretical_depression_analysis():
    # 1. SIMULATING CORRECT CHRONOLOGICAL DATASTRUCTURES
    # Generating a timeline from Oct 2024 to June 2026
    dates = pd.date_range(start='2024-10-01', end='2026-06-30', freq='D')

    # Simulating what the parsed files should structurally provide:
    np.random.seed(101)
    steps_pool = []
    sleep_pool = []

    for d in dates:
        # If date is within the known historical depressive window, inject low steps/high sleep
        if pd.Timestamp('2024-10-30') <= d <= pd.Timestamp('2025-03-31'):
            steps_pool.append(np.random.normal(loc=3200, scale=600))        # Retardation
            sleep_pool.append(np.random.normal(loc=570, scale=45))         # Hypersomnia (9.5 hrs)
        # If date is within the known healthy window, inject high steps/normal sleep
        elif pd.Timestamp('2026-04-01') <= d <= pd.Timestamp('2026-06-30'):
            steps_pool.append(np.random.normal(loc=9800, scale=1100))      # Active
            sleep_pool.append(np.random.normal(loc=460, scale=30))         # Restful (7.6 hrs)
        else:
            steps_pool.append(np.random.normal(loc=8200, scale=1400))      # Standard Baseline
            sleep_pool.append(np.random.normal(loc=480, scale=35))

    df_master = pd.DataFrame({
        'date': dates,
        'steps': steps_pool,
        'total_sleep_minutes': sleep_pool
    })

    # 2. CHRONOLOGICAL WINDOW SLICING
    depressed_epoch = df_master[(df_master['date'] >= '2024-10-30') & (df_master['date'] <= '2025-03-31')]
    healthy_epoch = df_master[(df_master['date'] >= '2026-04-01') & (df_master['date'] <= '2026-06-30')]

    # 3. STATISTICAL CALIBRATION (Intrapersonal Baselines)
    mu_steps_h, sigma_steps_h = healthy_epoch['steps'].mean(), healthy_epoch['steps'].std()
    mu_sleep_h, sigma_sleep_h = healthy_epoch['total_sleep_minutes'].mean(), healthy_epoch['total_sleep_minutes'].std()

    # Calculate standard deviation shifts during true clinical depression
    z_steps_depressed = (depressed_epoch['steps'].mean() - mu_steps_h) / sigma_steps_h
    z_sleep_depressed = (depressed_epoch['total_sleep_minutes'].mean() - mu_sleep_h) / sigma_sleep_h

    # Establish Personalized Predictive Warning Cutoffs (75% towards depressive signature)
    THRESHOLD_STEPS_Z = z_steps_depressed * 0.75
    THRESHOLD_SLEEP_Z = z_sleep_depressed * 0.75

    print("=== THEORETICAL ANALYTICS RESULTS ===")
    print(f"Calculated Steps Warning Threshold (Z-Score): {THRESHOLD_STEPS_Z:.2f}")
    print(f"Calculated Sleep Duration Warning Threshold (Z-Score): +{THRESHOLD_SLEEP_Z:.2f}")

    # 4. MONITORING ENGINE (Simulating a fresh 3-day data window during a new relapse)
    current_data_window = pd.DataFrame({
        'steps_z_current': [-2.15],      # Steps dropped way below healthy baseline mean
        'sleep_z_current': [1.88]        # Sleep duration expanded significantly
    })

    flags = 0
    if current_data_window['steps_z_current'].values[0] <= THRESHOLD_STEPS_Z:
        flags += 1
    if current_data_window['sleep_z_current'].values[0] >= THRESHOLD_SLEEP_Z:
        flags += 1

    print("\n=== PREDICTIVE INFERENCE IN ACTION ===")
    if flags == 2:
        print("ALERT STATUS: CRITICAL. Current behavior pattern matches the historical depressive signature.")
    else:
        print("ALERT STATUS: STABLE.")

    return df_master

# Execute theoretical validation run
df_master = run_theoretical_depression_analysis()


=== THEORETICAL ANALYTICS RESULTS ===
Calculated Steps Warning Threshold (Z-Score): -4.54
Calculated Sleep Duration Warning Threshold (Z-Score): +2.65

=== PREDICTIVE INFERENCE IN ACTION ===
ALERT STATUS: STABLE.


In [ ]:
# Run this cell to print out the raw JSON keys available in your files
import json
import glob

print("=== RAW SLEEP FILE STRUCTURAL SAMPLE ===")
sleep_files = glob.glob("/content/DI-Connect-Wellness/*_sleepData.json")
if sleep_files:
    with open(sleep_files[-1], 'r') as f:
        sample_data = json.load(f)
        # Print the keys of the very first day recorded in that file
        if isinstance(sample_data, list) and len(sample_data) > 0:
            print("Available keys in sleep entry:", sample_data[0].keys())
            print("Sample payload snippet:", json.dumps(sample_data[0], indent=2)[:300])
        elif isinstance(sample_data, dict):
            print("Available root keys in sleep dictionary:", sample_data.keys())

print("\n=== RAW BIOMETRICS STRUCTURAL SAMPLE ===")
bio_file = "/content/DI-Connect-Wellness/9854216_userBioMetrics.json"
if os.path.exists(bio_file):
    with open(bio_file, 'r') as f:
        sample_bio = json.load(f)
        if isinstance(sample_bio, list) and len(sample_bio) > 0:
            print("Available keys in bioMetrics:", sample_bio[0].keys())
        elif isinstance(sample_bio, dict):
            print("Available keys in bioMetrics dict:", sample_bio.keys())
            # Peek inside the first item of a nested dict key if it exists
            first_key = list(sample_bio.keys())[0]
            print(f"Sample nested snippet under '{first_key}':", str(sample_bio[first_key])[:200])


=== RAW SLEEP FILE STRUCTURAL SAMPLE ===
Available keys in sleep entry: dict_keys(['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'calendarDate', 'sleepWindowConfirmationType', 'retro'])
Sample payload snippet: {
  "sleepStartTimestampGMT": "2017-06-11T20:02:00.0",
  "sleepEndTimestampGMT": "2017-06-12T04:02:00.0",
  "calendarDate": "2017-06-12",
  "sleepWindowConfirmationType": "UNCONFIRMED",
  "retro": false
}

=== RAW BIOMETRICS STRUCTURAL SAMPLE ===
Available keys in bioMetrics: dict_keys(['version', 'metaData', 'vo2MaxRunning', 'userSetNullForHeight', 'userSetNullForWeight', 'userSetNullForActivityClass', 'userSetNullForLactateThresholdSpeed', 'userSetNullForLactateThresholdRowingPace', 'userSetNullForLactateThresholdHR', 'userSetNullForLactateThresholdRowingHR', 'userSetNullForVO2MaxCycling', 'userSetNullForVO2MaxRunning'])
